# Cartridge — Full Vector Extraction

Extracts Q, K, V attention vectors from **Llama 3.1 8B** for all 3 self-study datasets (3000 conversations total).

**Datasets:**
| Name | ~Tokens | Conversations |
|------|---------|---------------|
| multi_doc_qa_sanofi | 23K | 1000 |
| us_presidential_debates | ~49K | 1000 |
| apple_samsung_financials | ~119K | 1000 |

**Pipeline per dataset:**
1. Phase 1: Forward pass on context → extract K/V for 5 heads (once)
2. Phase 2: For each conversation → generate answer → forward pass → extract QA-portion Q/K/V

**Estimated time:** ~4h per dataset on A100 (~15s/conversation)
**Estimated storage:** ~7 GB total (context K/V + 3000 conversations)

**Resumable:** Skips already-extracted conversations. Safe to interrupt and restart.

**Upload:** Before running, upload the `self_study_conversations.jsonl` files for all 3 datasets.

In [ ]:
# ── 1. Setup + Config ──
import torch, os, gc, json, time
import numpy as np
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

assert torch.cuda.is_available(), "No GPU found!"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu} ({vram:.0f} GB)")

MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B"
NUM_Q_HEADS = 32
NUM_KV_HEADS = 8
HEAD_DIM = 128
MAX_NEW_TOKENS = 256
MAX_CONTEXT_TOKENS = 125000

SELECTED_HEADS = [
    {"layer": 0,  "q_head": 22, "kv_head": 5, "label": "p100_highest"},
    {"layer": 12, "q_head": 25, "kv_head": 6, "label": "p25"},
    {"layer": 15, "q_head": 22, "kv_head": 5, "label": "p50_median"},
    {"layer": 27, "q_head": 7,  "kv_head": 1, "label": "p75"},
    {"layer": 31, "q_head": 14, "kv_head": 3, "label": "p0_lowest"},
]

TARGET_LAYERS = sorted(set(h["layer"] for h in SELECTED_HEADS))
PER_LAYER_HEADS = {}
for h in SELECTED_HEADS:
    li = h["layer"]
    if li not in PER_LAYER_HEADS:
        PER_LAYER_HEADS[li] = ([], [])
    q_list, kv_list = PER_LAYER_HEADS[li]
    if h["q_head"] not in q_list:
        q_list.append(h["q_head"])
    if h["kv_head"] not in kv_list:
        kv_list.append(h["kv_head"])

# Only run apple_samsung_financials (sanofi + debates already done)
DATASETS = {
    # "multi_doc_qa_sanofi":      {"lb_idx": 181, "desc": "Sanofi Dupixent press releases"},
    # "us_presidential_debates":  {"lb_idx": 225, "desc": "2024 Biden-Trump debate transcripts"},
    "apple_samsung_financials": {"lb_idx": 135, "desc": "Apple & Samsung 10-K filings"},
}

# Output root
OUT_ROOT = Path("/content/cartridge_vectors")
OUT_ROOT.mkdir(exist_ok=True)

print(f"Layers: {TARGET_LAYERS}")
print(f"Datasets: {list(DATASETS.keys())}")
print(f"Output: {OUT_ROOT}")

In [ ]:
# ── 2. Load Model ──
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading {MODEL_NAME}...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)
model.eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
device = next(model.parameters()).device

print(f"Loaded in {time.time()-t0:.0f}s on {device}")
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.1f} / {vram:.0f} GB")

In [ ]:
# ── 3. Load Corpora from HuggingFace ──
from datasets import load_dataset

ds = load_dataset("THUDM/LongBench-v2", split="train")

corpora = {}
for name, cfg in DATASETS.items():
    row = ds[cfg["lb_idx"]]
    corpus = row["context"]
    tokens = tokenizer.encode("Context: " + corpus, add_special_tokens=True)
    if len(tokens) > MAX_CONTEXT_TOKENS:
        tokens = tokens[:MAX_CONTEXT_TOKENS]
    corpora[name] = {"corpus": corpus, "tokens": tokens}
    print(f"{name}: {len(corpus):,} chars -> {len(tokens):,} tokens")

del ds
gc.collect()

## Upload Self-Study Conversations

Upload the 3 JSONL files. Three options:

**Option A — Colab file browser (easiest):** Click the folder icon on the left sidebar, then drag & drop the 3 files into `/content/`. Rename them to:
- `multi_doc_qa_sanofi.jsonl`
- `us_presidential_debates.jsonl`
- `apple_samsung_financials.jsonl`

**Option B — `files.upload()`:** Run the next cell, it will prompt you to select files.

**Option C — Google Drive:** Mount drive and copy the files to `/content/`.

The files are at `cartridge/datasets/{name}/self_study_conversations.jsonl` on your local machine.

In [ ]:
# ── 4. Load Conversations ──
# Searches multiple locations for the JSONL files.
# Rename your files to {dataset_name}.jsonl before uploading to /content/.

# Optional: uncomment to use files.upload() dialog
# from google.colab import files; uploaded = files.upload()

SEARCH_PATHS = [
    # Uploaded to /content/ with dataset name
    "/content/{name}.jsonl",
    # Uploaded with original filename into a subfolder
    "/content/{name}/self_study_conversations.jsonl",
    # Google Drive mount
    "/content/drive/MyDrive/cartridge/{name}/self_study_conversations.jsonl",
    "/content/drive/MyDrive/{name}.jsonl",
    # Local (non-Colab)
    "cartridge/datasets/{name}/self_study_conversations.jsonl",
]

conversations = {}
for name in DATASETS:
    jsonl_path = None
    for pattern in SEARCH_PATHS:
        candidate = Path(pattern.format(name=name))
        if candidate.exists():
            jsonl_path = candidate
            break

    if jsonl_path is None:
        print(f"WARNING: {name} — no JSONL found. Searched:")
        for pattern in SEARCH_PATHS:
            print(f"  {pattern.format(name=name)}")
        continue

    convos = []
    with open(jsonl_path) as f:
        for line in f:
            if line.strip():
                convos.append(json.loads(line))
    conversations[name] = convos
    print(f"{name}: {len(convos)} conversations from {jsonl_path}")

print(f"\nTotal: {sum(len(v) for v in conversations.values())} conversations "
      f"across {len(conversations)} datasets")

In [ ]:
# ── 5. Extraction Helpers ──

def _get_rope_embeddings(rotary_emb, seq_len, dev):
    position_ids = torch.arange(seq_len, device=dev).unsqueeze(0)
    dummy = torch.zeros(1, 1, seq_len, 1, device=dev, dtype=torch.bfloat16)
    cos, sin = rotary_emb(dummy, position_ids)
    return cos, sin

def _apply_rotary_pos_emb(q, k, cos, sin):
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

def extract_qkv_from_hidden(hidden_states, layer_idx, seq_len):
    """Extract Q/K/V with RoPE. Returns detached bfloat16 CPU tensors."""
    layer = model.model.layers[layer_idx]
    attn = layer.self_attn
    ln = layer.input_layernorm

    with torch.no_grad():
        h = hidden_states.to(device)
        h_normed = ln(h)
        q_raw = attn.q_proj(h_normed).view(1, seq_len, NUM_Q_HEADS, HEAD_DIM).transpose(1, 2)
        k_raw = attn.k_proj(h_normed).view(1, seq_len, NUM_KV_HEADS, HEAD_DIM).transpose(1, 2)
        v_all = attn.v_proj(h_normed).view(1, seq_len, NUM_KV_HEADS, HEAD_DIM).transpose(1, 2)

        rotary = getattr(attn, "rotary_emb", None) or getattr(model.model, "rotary_emb", None)
        cos, sin = _get_rope_embeddings(rotary, seq_len, device)
        q_rope, k_rope = _apply_rotary_pos_emb(q_raw, k_raw, cos, sin)

    q_heads, kv_heads = PER_LAYER_HEADS[layer_idx]
    tensors = {}
    for hi in q_heads:
        tensors[f"Q_rope_head{hi}"] = q_rope[0, hi].cpu().to(torch.bfloat16)
        tensors[f"Q_raw_head{hi}"] = q_raw[0, hi].cpu().to(torch.bfloat16)
    for ki in kv_heads:
        tensors[f"K_rope_kvhead{ki}"] = k_rope[0, ki].cpu().to(torch.bfloat16)
        tensors[f"K_raw_kvhead{ki}"] = k_raw[0, ki].cpu().to(torch.bfloat16)
        tensors[f"V_kvhead{ki}"] = v_all[0, ki].cpu().to(torch.bfloat16)

    del h, h_normed, q_raw, k_raw, v_all, q_rope, k_rope
    torch.cuda.empty_cache()
    return tensors

def forward_with_hooks(input_ids, layers):
    """Forward pass capturing hidden states at target layers."""
    captured = {}
    hooks = []
    def make_hook(li):
        def hook_fn(module, args, output):
            captured[li] = args[0].detach().cpu()
        return hook_fn
    for li in layers:
        hooks.append(model.model.layers[li].register_forward_hook(make_hook(li)))
    with torch.no_grad():
        model.model(input_ids, use_cache=False)
    for h in hooks:
        h.remove()
    return captured

print("Helpers defined.")

## Run Extraction

One cell does everything — loops over datasets and conversations. Prints progress every 10 conversations. Resumable: skips conversations that already have .pt + example.json files.

In [ ]:
# ── 6. Full Extraction Loop ──
# After each dataset finishes, it is immediately tarred and downloaded.

import subprocess, shutil

tar_dir = Path("/content/tars")
tar_dir.mkdir(exist_ok=True)
staging = Path("/content/tar_staging")

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def tar_and_download(ds_name):
    """Tar one dataset's vectors and trigger download."""
    ds_dir = OUT_ROOT / ds_name
    tar_path = tar_dir / f"{ds_name}_vectors.tar"

    if tar_path.exists():
        print(f"  Tar already exists: {tar_path.name}")
        return

    # Stage as {name}/vectors/... so untar lands correctly
    stage_dir = staging / ds_name / "vectors"
    if stage_dir.exists():
        shutil.rmtree(stage_dir)
    shutil.copytree(ds_dir, stage_dir)

    subprocess.run(["tar", "cf", str(tar_path), ds_name], cwd=str(staging), check=True)
    shutil.rmtree(staging / ds_name)
    if staging.exists() and not any(staging.iterdir()):
        staging.rmdir()

    sz = tar_path.stat().st_size / 1e9
    print(f"  Tarred: {tar_path.name} ({sz:.2f} GB)")

    if IN_COLAB:
        print(f"  Downloading {tar_path.name}...")
        colab_files.download(str(tar_path))

# ═══════════════════════════════════════════════════
# Main loop
# ═══════════════════════════════════════════════════

grand_total = 0
grand_errors = 0
grand_t0 = time.time()

for ds_name in DATASETS:
    if ds_name not in conversations:
        print(f"\nSkipping {ds_name} (no conversations loaded)")
        continue

    convos = conversations[ds_name]
    context_tokens = corpora[ds_name]["tokens"]
    context_len = len(context_tokens)
    corpus = corpora[ds_name]["corpus"]

    ds_dir = OUT_ROOT / ds_name
    ctx_dir = ds_dir / "context"
    conv_dir = ds_dir / "conversations"

    print(f"\n{'='*70}")
    print(f"  {ds_name} — {context_len:,} context tokens, {len(convos)} conversations")
    print(f"{'='*70}")

    # ── Check existing conversations ──
    existing = set()
    if conv_dir.exists():
        for d in conv_dir.iterdir():
            if d.is_dir() and d.name.startswith("conv_"):
                if (d / "example.json").exists() and any(d.glob("layer_*.pt")):
                    existing.add(d.name)
    if existing:
        print(f"  Resuming: {len(existing)} already extracted")

    # ════════════════════════════════════════
    # Phase 1: Context K/V
    # ════════════════════════════════════════
    ctx_done = ctx_dir.exists() and any(ctx_dir.glob("layer_*.pt"))

    if not ctx_done:
        print(f"\n  Phase 1: Context K/V ({context_len:,} tokens)...")
        context_ids = torch.tensor([context_tokens], dtype=torch.long, device=device)
        t0 = time.time()
        captured = forward_with_hooks(context_ids, TARGET_LAYERS)
        print(f"  Forward: {time.time()-t0:.1f}s")

        for li in TARGET_LAYERS:
            hidden = captured.pop(li)
            tensors = extract_qkv_from_hidden(hidden, li, context_len)
            pt_path = ctx_dir / f"layer_{li:02d}.pt"
            pt_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(tensors, pt_path)
            del hidden, tensors

        with open(ctx_dir / "metadata.json", "w") as f:
            json.dump({"context_tokens": context_len, "context_chars": len(corpus),
                        "model": MODEL_NAME, "layers": TARGET_LAYERS, "heads": SELECTED_HEADS}, f, indent=2)

        del context_ids, captured
        gc.collect(); torch.cuda.empty_cache()
        ctx_mb = sum(p.stat().st_size for p in ctx_dir.glob("*.pt")) / 1e6
        print(f"  Context saved: {ctx_mb:.1f} MB")
    else:
        print(f"  Phase 1: Context already extracted, skipping.")

    # ════════════════════════════════════════
    # Phase 2: Conversations
    # ════════════════════════════════════════
    print(f"\n  Phase 2: {len(convos)} conversations...")
    extracted = 0
    skipped = 0
    errors = 0
    t_start = time.time()

    for ci, conv in enumerate(convos):
        conv_name = f"conv_{ci:04d}"
        conv_out = conv_dir / conv_name

        if conv_name in existing:
            skipped += 1
            extracted += 1
            continue

        question = conv["question"]
        qa_text = "\n\nQuestion: " + question + "\n\nAnswer:"
        qa_tokens = tokenizer.encode(qa_text, add_special_tokens=False)
        full_tokens = context_tokens + qa_tokens
        qa_start = context_len

        # Generate answer
        full_ids = torch.tensor([full_tokens], dtype=torch.long, device=device)
        try:
            with torch.no_grad():
                gen_output = model.generate(
                    full_ids, max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False, attention_mask=torch.ones_like(full_ids),
                )
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if "out of memory" in str(e).lower():
                print(f"    [{ci}] OOM gen, skip")
                gc.collect(); torch.cuda.empty_cache()
                errors += 1; continue
            raise

        answer_ids = gen_output[0, len(full_tokens):].tolist()
        answer_text = tokenizer.decode(answer_ids, skip_special_tokens=True)
        total_len = gen_output.shape[1]

        # Forward pass for extraction
        all_ids = gen_output.clone()
        del full_ids, gen_output
        gc.collect(); torch.cuda.empty_cache()

        try:
            captured = forward_with_hooks(all_ids, TARGET_LAYERS)
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if "out of memory" in str(e).lower():
                print(f"    [{ci}] OOM fwd, skip")
                gc.collect(); torch.cuda.empty_cache()
                errors += 1; continue
            raise

        # Extract QA-portion vectors — .clone() to avoid saving full backing storage
        for li in TARGET_LAYERS:
            hidden = captured.pop(li)
            tensors_full = extract_qkv_from_hidden(hidden, li, total_len)
            tensors_qa = {name: t[qa_start:].clone() for name, t in tensors_full.items()}
            pt_path = conv_out / f"layer_{li:02d}.pt"
            pt_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(tensors_qa, pt_path)
            del hidden, tensors_full, tensors_qa

        del all_ids, captured
        gc.collect(); torch.cuda.empty_cache()

        # Save metadata
        heads_label = ",".join(f"L{h['layer']}H{h['q_head']}" for h in SELECTED_HEADS)
        meta = {
            "example_id": f"{ds_name}_conv_{ci:04d}",
            "conversation_id": ci,
            "self_study_id": conv["id"],
            "seed_type": conv["seed_type"],
            "chunk_idx": conv["chunk_idx"],
            "question": question[:200],
            "generated_answer": answer_text[:200],
            "context_tokens": context_len,
            "qa_start": qa_start,
            "question_tokens": len(qa_tokens),
            "answer_tokens": len(answer_ids),
            "total_tokens": total_len,
            "sequence_length": total_len - qa_start,
            "heads_extracted": heads_label,
            "rope_included": True, "raw_included": True, "backend": "cuda",
        }
        with open(conv_out / "example.json", "w") as f:
            json.dump(meta, f, indent=2)

        extracted += 1

        if (extracted - skipped) % 10 == 0 and (extracted - skipped) > 0:
            elapsed = time.time() - t_start
            done = extracted - skipped
            rate = done / elapsed * 60
            remaining = len(convos) - extracted
            eta = remaining / rate if rate > 0 else 0
            mem = torch.cuda.memory_allocated() / 1e9
            print(f"    [{extracted}/{len(convos)}] {rate:.1f}/min, "
                  f"ETA {eta:.0f}min, GPU {mem:.1f}GB, err={errors}")

    elapsed = time.time() - t_start

    # Save dataset metadata
    ds_meta = {
        "task": f"self_study_{ds_name}",
        "source": "THUDM/LongBench-v2 + self-study",
        "model": MODEL_NAME,
        "selected_heads": SELECTED_HEADS,
        "layers_extracted": TARGET_LAYERS,
        "context_tokens": context_len,
        "n_conversations": extracted,
        "n_examples": extracted,
        "extraction_config": {"max_context_tokens": MAX_CONTEXT_TOKENS,
                              "max_new_tokens": MAX_NEW_TOKENS,
                              "store_raw_vectors": True, "store_rope_vectors": True},
    }
    with open(ds_dir / "metadata.json", "w") as f:
        json.dump(ds_meta, f, indent=2)

    print(f"\n  Done: {extracted} extracted ({skipped} resumed), "
          f"{errors} errors, {elapsed/60:.1f}min")
    grand_total += extracted
    grand_errors += errors

    # ════════════════════════════════════════
    # Tar + Download immediately
    # ════════════════════════════════════════
    print(f"\n  Packaging {ds_name}...")
    tar_and_download(ds_name)

grand_elapsed = time.time() - grand_t0
print(f"\n{'='*70}")
print(f"ALL DONE: {grand_total} conversations, {grand_errors} errors, "
      f"{grand_elapsed/60:.1f}min total")
print(f"{'='*70}")

In [ ]:
# ── 7. Verify + Summary ──
print("Storage summary:\n")

total_bytes = 0
for ds_name in DATASETS:
    ds_dir = OUT_ROOT / ds_name
    if not ds_dir.exists():
        continue

    ctx_bytes = sum(p.stat().st_size for p in (ds_dir / "context").glob("*.pt")) if (ds_dir / "context").exists() else 0
    conv_bytes = sum(p.stat().st_size for p in ds_dir.rglob("conversations/**/*.pt")) if (ds_dir / "conversations").exists() else 0
    n_convos = len(list((ds_dir / "conversations").iterdir())) if (ds_dir / "conversations").exists() else 0
    ds_total = ctx_bytes + conv_bytes

    sample_ok = ""
    if n_convos > 0:
        sample_dir = ds_dir / "conversations" / "conv_0000"
        if sample_dir.exists():
            sample_mb = sum(p.stat().st_size for p in sample_dir.glob("*.pt")) / 1e6
            sample_ok = f" (sample conv: {sample_mb:.2f} MB)"

    total_bytes += ds_total
    print(f"  {ds_name}:")
    print(f"    Context:  {ctx_bytes/1e6:.1f} MB")
    print(f"    {n_convos} convos: {conv_bytes/1e6:.1f} MB{sample_ok}")
    print(f"    Total:    {ds_total/1e6:.1f} MB")
    print()

print(f"Grand total: {total_bytes/1e9:.2f} GB")

# Show tars
print(f"\nDownloaded tars:")
for tar in sorted(tar_dir.glob("*.tar")):
    print(f"  {tar.name}: {tar.stat().st_size/1e9:.2f} GB")

print(f"\nTo reassemble locally:")
print(f"  cd cartridge/datasets")
print(f"  for f in *_vectors.tar; do tar xf \"$f\"; done")

In [ ]:
# ── 8. Re-download all tars ──
# Use this cell to re-trigger downloads if the browser blocked them
# or if you need to download them again on a fresh session.

from google.colab import files as colab_files
from pathlib import Path

tar_dir = Path("/content/tars")

tar_files = sorted(tar_dir.glob("*.tar"))
if not tar_files:
    print("No tar files found in /content/tars/. Run extraction first.")
else:
    print(f"Found {len(tar_files)} tar file(s):\n")
    for tar in tar_files:
        sz = tar.stat().st_size / 1e9
        print(f"  Downloading {tar.name} ({sz:.2f} GB)...")
        colab_files.download(str(tar))
    print("\nAll downloads triggered. Check your browser's download bar.")